# RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
# Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 1-understand-your-setup.pdf
  ✓ Loaded 9 pages

Processing: 2-define-your-agests-identity.pdf
  ✓ Loaded 6 pages

Processing: 3-handson-transform-your-agent.pdf
  ✓ Loaded 6 pages

Total documents loaded: 21


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-12-17T04:17:07+00:00', 'moddate': '2025-12-17T04:17:07+00:00', 'source': '..\\data\\pdf\\1-understand-your-setup.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '1-understand-your-setup.pdf', 'file_type': 'pdf'}, page_content='Understand\nyour setup\nLanguage support\nAgent Development Kit (ADK) supports multiple \nprogramming languages:\nPython 3.11+ (covered in this course)\nJava 17+ (see Java Quickstart)\nThis course uses Python for examples. The \nconcepts apply to all supported languages.\nBefore you begin, ensure you have:\nRequired software (Python)\nPython 3.11 or higher – ADK requires \u2028\nPython 3.11+\nCheck your version:  \u2028\nor \nIf needed, download from python.org\npip – Python package installer \u2028\n(included with Python)\nVerify:  or \u2028\nTerminal/command prompt – Access to \u202

In [6]:
# Text splitting get into Chuncks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks = split_documents(all_pdf_documents)
chunks

Split 21 documents into 36 chunks

Example chunk:
Content: Understand
your setup
Language support
Agent Development Kit (ADK) supports multiple 
programming languages:
Python 3.11+ (covered in this course)
Java 17+ (see Java Quickstart)
This course uses Pytho...
Metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-12-17T04:17:07+00:00', 'moddate': '2025-12-17T04:17:07+00:00', 'source': '..\\data\\pdf\\1-understand-your-setup.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '1-understand-your-setup.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-12-17T04:17:07+00:00', 'moddate': '2025-12-17T04:17:07+00:00', 'source': '..\\data\\pdf\\1-understand-your-setup.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '1-understand-your-setup.pdf', 'file_type': 'pdf'}, page_content='Understand\nyour setup\nLanguage support\nAgent Development Kit (ADK) supports multiple \nprogramming languages:\nPython 3.11+ (covered in this course)\nJava 17+ (see Java Quickstart)\nThis course uses Python for examples. The \nconcepts apply to all supported languages.\nBefore you begin, ensure you have:\nRequired software (Python)\nPython 3.11 or higher – ADK requires \u2028\nPython 3.11+\nCheck your version:  \u2028\nor \nIf needed, download from python.org\npip – Python package installer \u2028\n(included with Python)\nVerify:  or \u2028\nTerminal/command prompt – Access to \u202

## Embeding and VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\jonas\OneDrive\Área de Trabalho\AI Eng Courses\Complete Agentic AI Course\3-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\jonas\OneDrive\Área de Trabalho\AI Eng Courses\Complete Agentic AI Course\3-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jonas\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 1

Model loaded successfully. Embedding dimension: 384


C:\Users\jonas\AppData\Local\Temp\ipykernel_40112\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [11]:
chunks

[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-12-17T04:17:07+00:00', 'moddate': '2025-12-17T04:17:07+00:00', 'source': '..\\data\\pdf\\1-understand-your-setup.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '1-understand-your-setup.pdf', 'file_type': 'pdf'}, page_content='Understand\nyour setup\nLanguage support\nAgent Development Kit (ADK) supports multiple \nprogramming languages:\nPython 3.11+ (covered in this course)\nJava 17+ (see Java Quickstart)\nThis course uses Python for examples. The \nconcepts apply to all supported languages.\nBefore you begin, ensure you have:\nRequired software (Python)\nPython 3.11 or higher – ADK requires \u2028\nPython 3.11+\nCheck your version:  \u2028\nor \nIf needed, download from python.org\npip – Python package installer \u2028\n(included with Python)\nVerify:  or \u2028\nTerminal/command prompt – Access to \u202

In [ ]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 36 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.41s/it]


Generated embeddings with shape: (36, 384)
Adding 36 documents to vector store...
Successfully added 36 documents to vector store
Total documents in collection: 36


## Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [16]:
rag_retriever.retrieve("How to create an AI agent")

Retrieving documents for query: 'How to create an AI agent'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d9da2f4f_31',
  'content': "Complete transformed agent\nPython\nfrom google.adk.agents.llm_agent import Agent\n\nroot_agent = Agent(\n\xa0\xa0\xa0\xa0model='gemini-2.5-flash',\n\xa0\xa0\xa0\xa0name='math_tutor_agent',\n\xa0\xa0\xa0\xa0description='Helps students learn algebra by guiding them through problem-\nsolving steps.',\n\xa0\xa0\xa0\xa0instruction='You are a patient math tutor. Help students with algebra problems.'\n)\n3\nSave this to your  file in your  directory from module 1.\nThis simple agent demonstrates:\nSpecific role definition (math tutor)\nClear personality (patient)\nDefined task scope (algebra problems)\nAre you ready for more? \nCourse 3 will teach you how to enhance this with professional \u2028\ninstruction patterns, boundaries, and examples for production use.\nagent.py my_first_agent",
  'metadata': {'creationdate': '2025-12-17T04:21:06+00:00',
   'page': 2,
   'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)',
   'file_type': 'pdf',
   'co

In [17]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### RAG Pipeline - VectorDB to LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

In [22]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [23]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [24]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
    
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [25]:
# get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb Context pipeline with LLM output

In [33]:
# Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key,
               model_name = "openai/gpt-oss-20b",
               temperature = 0.1,
               max_tokens = 1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [34]:
answer = rag_simple("How to create an AI agent?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'How to create an AI agent?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**How to create an AI agent**

1. **Install the ADK** (if not already done)  
   ```bash
   pip install google-adk
   ```

2. **Create a Python file** (e.g., `agent.py`) in your project directory.

3. **Import the Agent class** and instantiate it with the required parameters:

   ```python
   from google.adk.agents.llm_agent import Agent

   # Define the agent
   root_agent = Agent(
       model='gemini-2.5-flash',          # LLM that does the reasoning
       name='root_agent',                 # Unique identifier
       description='A helpful assistant agent.',  # What it does
       instruction='You are a helpful assistant.' # How it behaves
   )
   ```

4. **(Optional) Add tools** if you need external capabilities, e.g.:

   ```python
   from google.adk.tools import Calculator

   root_agent.add_tool(Calculator())
   ```

5. **Save the file** and run it to test:

   ```bash
   python agent.py
   ```

6. **(Optional) Use the ADK CLI** to scaffold or manage agents:

   ```bash
   adk 

### Enhanced RAG Pipeline Features

In [39]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is ADK", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is ADK'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: ADK is the **Agent Development Kit**—a set of command‑line tools that discover, run, and manage your agent code. It looks for a Python variable named `root_agent` as the entry point to your agent system.
Sources: [{'source': '2-define-your-agests-identity.pdf', 'page': 4, 'score': 0.16254836320877075, 'preview': '# Variable name that ADK tools look for (must be root_agent)\nroot_agent = my_specialized_agent\nWhy root_agent?\nADK command-line tools look for a Python variable named root_agent as the entry point to your agent \nsystem. This is a convention that allows ADK to discover and run your agent.\nFrom ADK do...'}]
Confidence: 0.16254836320877075
Context Preview: # Variable name that ADK tools look for (must be root_agent)
root_agent = my_specialized_agent
Why root_agent?
ADK command-line tools look for a Python variable named root_agent as the entry point to your agent 
system. This is a convention that allows ADK to discover and run your agent.
From ADK do


In [38]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is ADK?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is ADK?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.28it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
# Variable name that ADK tools look for (must be root_agent)
root_agent = my_specialized_ag

ent
Why root_agent?
ADK command-line tools look for a Python variable named root_agent as the entry point to your agent 
system. This is a convention that allows ADK to discover and run your agent.
From ADK docs:
“The agent.py file contains a root_agent definition, which is the only required element of an ADK agent.”
Can the internal name be different from the variable name?
Y es. The  parameter inside  is separate from the variable name: name Agent()
K ey r u le :  Always assign your main agent to a variable named , so ADK tools can find it.
I n module 3 , you ’ ll learn how ADK tools use root_agent to run your agent in di ff erent ways.
root_agent

Question: what is ADK?

Answer:

Final Answer: ADK is the **Agent Development Kit** – a set of command‑line tools that discover and run your agent by looking for a Python variable named `root_agent`.

Citations:
[1] 2-define-your-agests-identity.pdf (page 4)
Summary: ADK stands for Agent Development Kit, a collection of command‑line tools 